# Notebook 00 — Context

**Purpose:** establish the premise of context for one research paper.

This notebook does not try to solve the whole paper. It creates the first stable object:

- paper metadata
- abstract-level context
- initial claims
- first lab-report context paragraph
- saved JSON result

Notebook 01 will decompose the paper more deeply.

## 0. Setup

Run this from the repo root, or adjust the path below.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from accurate_intelligence.papers import load_arxiv
from accurate_intelligence.extraction import extract_claims
from accurate_intelligence.schemas import LabReport
from accurate_intelligence.reporting import build_markdown, save_markdown_report
from accurate_intelligence.utils import ensure_dir, save_json
from accurate_intelligence.paths import RESULTS, REPORTS

ensure_dir(RESULTS)
ensure_dir(REPORTS)

print("Project root:", PROJECT_ROOT)
print("Results:", RESULTS)
print("Reports:", REPORTS)

## 1. Choose paper

Default: ResearchClawBench paper.

In [ ]:
ARXIV_ID = "2606.07591"
paper = load_arxiv(ARXIV_ID)

paper

## 2. Paper metadata

In [ ]:
print("Title:", paper.title)
print("Authors:", ", ".join(paper.authors[:8]))
if len(paper.authors) > 8:
    print(f"... plus {len(paper.authors) - 8} more")
print("\nAbstract:\n")
print(paper.abstract)

## 3. Initial claim extraction

This is deliberately simple in v0: split the abstract into claim-like sentences. Later notebooks can replace this with stronger parsing.

In [ ]:
claims = extract_claims(paper)

for i, claim in enumerate(claims, start=1):
    print(f"{i}. {claim.text}")

## 4. Premise of context

Notebook 00 writes the first plain-language context statement. This is the seed that later notebooks revise, verify, and score.

In [ ]:
context = f"""
{paper.title} asks whether AI systems can perform end-to-end scientific research rather than merely summarize or discuss research papers. The initial context is evaluation: given a scientific task, raw data, and related literature, the system must produce a report that can be compared against a target paper. For accurate-intelligence, this establishes the baseline distinction between generated research text and verified scientific reconstruction.
""".strip()

print(context)

## 5. Build first LabReport object

In [ ]:
report = LabReport(
    context=context,
    observations=[claim.text for claim in claims[:5]],
    prescriptions=[
        "Use Notebook 01 to decompose the target paper into objective, protocol, evidence, figures, and scientific core.",
        "Use Notebook 31 later to compare the generated lab report against ResearchClaw-style scoring.",
        "Use CGCS later to track whether revisions preserve constraints across papers."
    ],
)

markdown = build_markdown(report)
print(markdown)

## 6. Save results

In [ ]:
result = {
    "arxiv_id": ARXIV_ID,
    "title": paper.title,
    "authors": paper.authors,
    "abstract": paper.abstract,
    "context": report.context,
    "observations": report.observations,
    "prescriptions": report.prescriptions,
}

save_json(result, RESULTS / "00_context.json")
save_markdown_report(report, REPORTS / "00_context.md")

print("Saved:")
print(RESULTS / "00_context.json")
print(REPORTS / "00_context.md")

## 7. Next notebook handoff

Notebook 01 should read `results/00_context.json` and produce:

- objective
- protocol
- evidence map
- figure/table inventory
- scientific core draft